In [0]:
import importlib
from config.config import *

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta").load(
    f"{bronze_path}/bronze_un_tourism_inbound_arrivals_raw"
)
display(df.head(10))

In [0]:
header_row = df.first()

new_columns = [
    str(header_row[c]).strip()
    for c in df.columns
]

df = df.filter(
    col(df.columns[0]) != header_row[df.columns[0]]
)

df = df.toDF(*new_columns)

In [0]:
df = df.drop(
    "indicator_previous_code"
)

In [0]:
df = df.withColumnRenamed(
    "reporter_area_label",
    "reporter_area"
).withColumnRenamed(
    "partner_area_label",
    "partner_area"
).withColumnRenamed(
    "flag",
    "observation_flag"
)

In [0]:
string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

for c in string_columns:
    df = df.withColumn(
        c,
        trim(col(c))
    )


In [0]:
df = df.withColumn(
    "reporter_area_code",
    expr("try_cast(reporter_area_code AS INT)")
).withColumn(
    "partner_area_code",
    expr("try_cast(partner_area_code AS INT)")
).withColumn(
    "year",
    expr("try_cast(year AS INT)")
).withColumn(
    "arrivals",
    expr("try_cast(value AS DOUBLE)")
)

df = df.drop("value")

In [0]:
df = df.withColumn(
    "arrivals",
    col("arrivals") * 1000
).withColumn(
    "unit",
    lit("NR")
)

In [0]:
df = df.withColumn(
    "reporter_geography_type",
    when(
        col("reporter_area").isNotNull(),
        "country"
    ).otherwise("unknown")
).withColumn(
    "partner_geography_type",
    when(
        col("partner_area") == "World",
        "world_aggregate"
    ).otherwise("country")
)

df.drop("notes")

silver_df = df.select(
    "indicator_code",
    "indicator_label",
    "reporter_area_code",
    "reporter_area",
    "reporter_geography_type",
    "partner_area_code",
    "partner_area",
    "partner_geography_type",
    "year",
    "arrivals",
    "unit",
    "observation_flag",
    "flag_label"
)


In [0]:
output_path = f"{silver_path}/un_tourism_inbound_arrivals"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_path)

print("Cleaning for un_tourism_inbound_arrivals completed")

In [0]:
spark.read.format("delta").load(output_path).display()